In [0]:
%sql
-- Replace catalog.schema.table as needed
CREATE OR REPLACE TABLE sts_prd.s_ipd.s_dimdate_analytics_ipd
USING DELTA
AS
WITH params AS (
  SELECT
    current_date() AS today,
    year(current_date()) AS today_year,
    month(current_date()) AS today_month,
    CASE WHEN month(current_date()) >= 10 THEN year(current_date()) + 1 ELSE year(current_date()) END AS current_fy,
    CASE WHEN month(current_date()) >= 10 THEN month(current_date()) - 9 ELSE month(current_date()) + 3 END AS current_fm
),
current_q AS (
  SELECT
    *,
    CASE
      WHEN current_fm BETWEEN 1 AND 3  THEN 1
      WHEN current_fm BETWEEN 4 AND 6  THEN 2
      WHEN current_fm BETWEEN 7 AND 9  THEN 3
      ELSE 4
    END AS current_fq
  FROM params
),
base_calendar AS (
  SELECT explode(sequence(to_date('2010-01-01'), to_date('2040-12-31'), INTERVAL 1 DAY)) AS d
),
helpers AS (
  SELECT
    d,
    -- Fiscal helpers (Oct–Sep)
    CASE WHEN month(d) >= 10 THEN year(d) + 1 ELSE year(d) END AS fy,
    CASE WHEN month(d) >= 10 THEN month(d) - 9 ELSE month(d) + 3 END AS fm,
    -- Quarter boundaries
    date_trunc('quarter', d) AS q_start,
    date_add(add_months(date_trunc('quarter', d), 3), -1) AS q_end,
    -- Year boundaries
    date_trunc('year', d) AS y_start,
    date_add(add_months(date_trunc('year', d), 12), -1) AS y_end,
    -- Month boundaries
    date_trunc('month', d) AS m_start,
    last_day(d) AS m_end,
    -- ISO week boundaries (Spark uses Monday as week start)
    date_trunc('week', d) AS iso_w_start,
    date_add(date_trunc('week', d), 6) AS iso_w_end,
    -- Fiscal year start (Oct 1 of the fiscal year)
    make_date(CASE WHEN month(d) >= 10 THEN year(d) ELSE year(d) - 1 END, 10, 1) AS fy_start
  FROM base_calendar
)
SELECT
  /* ==================== Calendar Columns ==================== */
  h.d                                             AS `CalendarDate`,
  year(h.d)                                       AS `YearCalendar`,
  month(h.d)                                      AS `MonthCalendar`,
  date_format(h.d, 'MMMM')                        AS `MonthNameCalendar`,
  date_format(h.d, 'yyyy-MM')                     AS `YearMonthCalendar`,
  substr(date_format(h.d,'MMMM'),1,1)             AS `MonthInitialCalendar`,
  date_format(h.d,'MMM')                          AS `MonthAbbrevCalendar`,
  concat('Q', quarter(h.d))                       AS `QuarterCalendar`,
  dayofweek(h.d)                                  AS `DayOfWeekCalendar`,     -- Sun=1..Sat=7
  date_format(h.d,'EEEE')                         AS `DayNameCalendar`,

  /* ==================== Fiscal Columns (Oct–Sep) ==================== */
  (((month(h.d) + 2) % 12) + 1)                   AS `FiscalMonthSort`,
  h.fy                                            AS `FiscalYear`,
  h.fm                                            AS `FiscalMonth`,
  concat('Qtr',
    CASE
      WHEN h.fm BETWEEN 1 AND 3 THEN 1
      WHEN h.fm BETWEEN 4 AND 6 THEN 2
      WHEN h.fm BETWEEN 7 AND 9 THEN 3
      ELSE 4
    END
  )                                               AS `FiscalQuarter`,
  concat(lpad(CAST(h.fy AS STRING),4,'0'), '-', lpad(CAST(h.fm AS STRING),2,'0'))
                                                  AS `FiscalYearMonth`,
  make_date(h.fy, h.fm, 1)                        AS `FiscalDate`,

  /* ==================== Surrogates & Offsets ==================== */
  CAST(date_format(h.d,'yyyyMMdd') AS INT)        AS `DateKeyDay`,
  CAST(date_format(h.d,'yyyyMM')   AS INT)        AS `DateKeyMonth`,
  CAST(date_format(h.d,'yyyy')     AS INT)        AS `DateKeyYear`,

  -- requested additional surrogate keys (aliases for convenience)
  CAST(date_format(h.d,'yyyy')     AS INT)        AS skeyYear,
  CAST(date_format(h.d,'yyyyMM')   AS INT)        AS skeyMonth,
  CAST(date_format(h.d,'yyyyMMdd') AS INT)        AS skeyDay,

  (h.fy - cq.current_fy)                          AS `FiscalYearOffset`,
  CAST(FLOOR(months_between(h.d, date_trunc('month', current_date()))) AS INT)
                                                  AS `FiscalMonthOffset`,

  /* ==================== ISO Week System ==================== */
  h.iso_w_start                                   AS `ISOWeekStart`,         -- Monday
  h.iso_w_end                                     AS `ISOWeekEnd`,
  weekofyear(h.d)                                 AS `ISOWeek`,
  CAST(date_format(h.d,'yyyy') AS INT)            AS `ISOWeekYear`,          -- week-based year

  /* ==================== Month / Quarter helpers ==================== */
  day(h.d)                                        AS `DayOfMonth`,
  dayofyear(h.d)                                  AS `DayOfYear`,
  h.m_end                                         AS `EndOfMonth`,
  h.m_start                                       AS `StartOfMonth`,
  h.q_start                                       AS `StartOfQuarter`,
  h.q_end                                         AS `EndOfQuarter`,
  h.y_start                                       AS `StartOfYear`,
  h.y_end                                         AS `EndOfYear`,
  CASE WHEN month(h.d) <= 6 THEN 1 ELSE 2 END     AS `HalfYear`,
  (h.m_end = h.d)                                 AS `IsMonthEnd`,
  (h.q_end = h.d)                                 AS `IsQuarterEnd`,
  (h.y_end = h.d)                                 AS `IsYearEnd`,
  CASE WHEN (mod(year(h.d),4)=0 AND (mod(year(h.d),100)<>0 OR mod(year(h.d),400)=0)) THEN TRUE ELSE FALSE END
                                                  AS `IsLeapYear`,
  day(h.m_end)                                    AS `DaysInMonth`,

  /* ==================== Calendar offsets ==================== */
  datediff(h.d, current_date())                   AS `CalendarDayOffset`,
  CAST(FLOOR(months_between(date_trunc('quarter', h.d), date_trunc('quarter', current_date()))/3) AS INT)
                                                  AS `CalendarQuarterOffset`,
  CAST(FLOOR(months_between(date_trunc('year', h.d), date_trunc('year', current_date()))/12) AS INT)
                                                  AS `CalendarYearOffset`,
  CAST(FLOOR(months_between(h.m_start, date_trunc('month', current_date()))) AS INT)
                                                  AS `CalendarMonthOffset`,

  /* ==================== Fiscal week (Oct–Sep) ==================== */
  h.fy_start                                      AS `FiscalYearStart`,
  (datediff(date_trunc('week', h.d), date_trunc('week', h.fy_start)) / 7) + 1
                                                  AS `FiscalWeek`,
  h.fy                                            AS `FiscalWeekYear`,
  date_trunc('week', h.d)                         AS `FiscalWeekStart`,
  date_add(date_trunc('week', h.d), 6)            AS `FiscalWeekEnd`,

  /* ==================== Business-day scaffolding ==================== */
  CAST(NULL AS BOOLEAN)                           AS `IsHoliday`,
  CAST(NULL AS STRING)                            AS `HolidayName`,
  (dayofweek(h.d) BETWEEN 2 AND 6)                AS `IsWeekday`,              -- Mon=2..Sun=1
  CAST(NULL AS BOOLEAN)                           AS `IsBusinessDay`,
  CAST(NULL AS INT)                               AS `BusinessDayOfMonth`,
  CAST(NULL AS INT)                               AS `BusinessDayOfYear`,

  /* ==================== Canonical date surrogate ==================== */
  CAST(date_format(h.d,'yyyyMMdd') AS INT)        AS date_sk,

  -- Calendar month grain (useful for other facts)
  CAST(date_format(h.d,'yyyyMM')   AS INT)        AS calendar_month_sk,

  -- Fiscal month grain as yyyymm (FYYYYY * 100 + FM) e.g., 202502 for FY2025 Feb
  (h.fy * 100 + h.fm)                              AS fiscal_month_sk,

  -- Fiscal period key to match Finance '2025005' (FY * 1000 + FM)
  (h.fy * 1000 + h.fm)                             AS fiscal_period_sk


FROM helpers h
CROSS JOIN current_q cq
;

-- (Optional) Governance & performance touch-ups
-- ALTER TABLE main.analytics.dim_date SET TBLPROPERTIES (
--   'owner'='IPD Analytics',
--   'sla'='daily rebuild',
--   'pii'='false',
--   'lineage'='generated in SQL from calendar sequence; fiscal Oct–Sep'
-- );
-- ALTER TABLE main.analytics.dim_date
--   ADD CONSTRAINT valid_range CHECK (date_sk BETWEEN 20200101 AND 20301231);
-- OPTIMIZE main.analytics.dim_date ZORDER BY (date_sk);
